In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import shap
from src.scoutai_common import (
    get_engine, load_master_view, engineer_features, get_model,
    FULL_FEATURES, PERFORMANCE_FEATURES,
)

NAME_MAP = {
    'remainder__age': 'Player Age', 'remainder__max_career_transfer_fee': 'Max Transfer Fee',
    'remainder__most_recent_transfer_fee': 'Most Recent Transfer Fee',
    'remainder__international_caps': 'International Caps',
    'remainder__contract_months_remaining': 'Contract Remaining',
    'remainder__total_appearances': 'Total Appearances', 'remainder__stadium_seats': 'Stadium Capacity',
    'remainder__minutes_per_match': 'Minutes Per Match', 'remainder__foreigners_percentage': 'Foreigner Ratio',
    'remainder__has_transfer_history': 'Transfer History', 'remainder__club_avg_age': 'Club Avg Age',
    'remainder__total_goals': 'Total Goals', 'remainder__goals_per_90': 'Goals Per 90',
    'remainder__international_goals': 'International Goals', 'remainder__club_squad_size': 'Squad Size',
    'remainder__total_assists': 'Total Assists', 'remainder__height_in_cm': 'Height (cm)',
    'remainder__wonderkid_hype': 'Wonderkid Hype Index', 'remainder__assists_per_90': 'Assists Per 90',
    'remainder__total_yellow_cards': 'Yellow Cards', 'remainder__total_red_cards': 'Red Cards',
    'remainder__league_coefficient': 'League Quality', 'cat__passport_tier_Tier_1': 'Passport (Tier 1)',
    'cat__detailed_position_Goalkeeper': 'Position: Goalkeeper',
}

def clean_names(feature_names):
    return [NAME_MAP.get(n, n.replace('remainder__', '').replace('cat__', '').replace('_', ' ').title())
            for n in feature_names]

engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)

print("Available players (first 10):", df['player_name'].head(10).tolist())
target_player = input("Enter the player name to analyze: ")
player_data = df[df['player_name'].str.contains(target_player, case=False, na=False)]

In [ ]:
if player_data.empty:
    print("\n[ERROR] Player not found. Please check the spelling.")
else:
    player = player_data.iloc[0]
    player_row = player_data.iloc[[0]]

    has_history = bool(player_row['has_transfer_history'].iloc[0])
    model_label = "full" if has_history else "performance_only"
    feature_list = FULL_FEATURES if has_history else PERFORMANCE_FEATURES

    print(f"[SYSTEM] has_transfer_history={int(has_history)} -> using '{model_label}' model.")

    pipeline = get_model(model_label)
    preprocessor = pipeline.named_steps['preprocessor']
    regressor = pipeline.named_steps['regressor']
    explainer = shap.TreeExplainer(regressor)
    names = clean_names(preprocessor.get_feature_names_out())

    X_player = player_row[feature_list]
    X_player_transformed = preprocessor.transform(X_player)
    shap_values_player = explainer(X_player_transformed)

    explanation = shap.Explanation(values=shap_values_player[0], base_values=explainer.expected_value,
                                    data=X_player_transformed[0], feature_names=names)

    plt.figure(figsize=(12, 8))
    shap.plots.waterfall(explanation, show=False)
    plt.title(f"ScoutAI Value Impact Analysis: {player['player_name']} ({model_label} model)",
              fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    safe_name = player['player_name'].replace(' ', '_')
    plt.savefig(f'../images/shap_waterfall_{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()